# Chapter 5: Design Consistent Hashing
*System Design Interview*

## TL;DR

Consistent hashing maps both servers and keys onto a circular hash ring so that adding or removing a server only redistributes **k/n** keys on average (instead of nearly all). **Virtual nodes** smooth out imbalanced partitions, and the technique underpins systems like DynamoDB, Cassandra, and Discord.

## Requirements

| Requirement | Detail |
|---|---|
| Even distribution | Requests/data spread uniformly across servers |
| Minimal disruption | Adding/removing a server moves the fewest keys possible |
| Horizontal scaling | Support dynamic addition and removal of nodes |
| Heterogeneous capacity | Larger servers can handle proportionally more keys |

## Estimation: Rehashing Impact

In [ ]:
# Compare key redistribution: modular hashing vs consistent hashing
num_keys = 1_000_000
num_servers_before = 4
num_servers_after = 3  # one server removed

# Modular hashing: nearly all keys remap
keys_remapped_modular = sum(
    1 for k in range(num_keys)
    if k % num_servers_before != k % num_servers_after
)
pct_modular = keys_remapped_modular / num_keys * 100

# Consistent hashing: ~k/n keys remap (one partition moves)
keys_remapped_consistent = num_keys / num_servers_before
pct_consistent = keys_remapped_consistent / num_keys * 100

print(f'Modular hashing: {keys_remapped_modular:,} keys remapped ({pct_modular:.1f}%)')
print(f'Consistent hashing: ~{int(keys_remapped_consistent):,} keys remapped ({pct_consistent:.1f}%)')
print(f'\nConsistent hashing reduces redistribution by ~{pct_modular/pct_consistent:.0f}x')

## Estimation: Virtual Nodes and Standard Deviation

In [ ]:
# Effect of virtual nodes on load distribution
import math

num_keys = 1_000_000
num_servers = 10

for vnodes in [1, 10, 50, 100, 200, 500]:
    total_points = num_servers * vnodes
    mean_keys_per_server = num_keys / num_servers
    # Std dev decreases roughly as 1/sqrt(vnodes)
    std_dev_pct = 100 / math.sqrt(vnodes)
    std_dev_keys = mean_keys_per_server * std_dev_pct / 100
    print(f'vnodes={vnodes:>3}  |  ring points={total_points:>5}  |  '
          f'std_dev ~ {std_dev_pct:5.1f}% of mean  |  '
          f'+/- {int(std_dev_keys):,} keys/server')

## High-Level Design

```mermaid
graph LR
    subgraph HashRing[Hash Ring]
        direction TB
        S0[Server 0] --> S1[Server 1]
        S1 --> S2[Server 2]
        S2 --> S3[Server 3]
        S3 --> S0
    end

    K0((key0)) -.-> S1
    K1((key1)) -.-> S2
    K2((key2)) -.-> S3
    K3((key3)) -.-> S0
```

## Deep Dive

### The Rehashing Problem
With modular hashing (`hash(key) % N`), changing `N` remaps nearly every key. This causes **cache miss storms** when servers join or leave.

### Hash Ring Mechanics
1. Map hash space `[0, 2^160)` into a circle (ring)
2. Place servers on the ring via `hash(server_ip)`
3. Place keys on the ring via `hash(key)`
4. Each key belongs to the first server found **clockwise**

### Virtual Nodes
- Each physical server gets **V** virtual positions on the ring
- More virtual nodes = smaller standard deviation in load
- Trade-off: more memory for virtual-node metadata
- Production systems typically use 100--200 virtual nodes

### Adding / Removing Servers
- **Add**: only keys in the arc between new node and its predecessor move
- **Remove**: only keys in the removed node's arc move to the next server clockwise

## Trade-offs

| Dimension | Benefit | Cost |
|---|---|---|
| Virtual nodes (high count) | More uniform distribution | More memory for node map |
| Virtual nodes (low count) | Less memory | Higher load variance |
| SHA-1 hash function | Large hash space, uniform | Computation overhead |
| Heterogeneous weights | Match server capacity | Complexity in management |

## Key Takeaways

1. Consistent hashing remaps only **k/n** keys vs nearly all with modular hashing
2. Virtual nodes are essential for real-world even distribution
3. The technique enables **horizontal scaling** with minimal data movement
4. Used in DynamoDB, Cassandra, Discord, Akamai CDN, Maglev load balancer

## Related Concepts

- [[rehashing-problem]] -- why simple modular hashing fails at scale
- [[hash-ring]] -- the circular data structure behind consistent hashing
- [[consistent-hashing]] -- the core algorithm
- [[virtual-nodes]] -- technique for balanced partitions
- [[key-redistribution]] -- mechanics of adding/removing servers